# Magnetic field of current loops (Biot–Savart)

*em · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/em/biot_savart)

The Biot–Savart law turns a wire into a sum over short segments. The cross product $d\mathbf l\times\mathbf r$ is the Levi-Civita tensor $\varepsilon_{ijk}$ contracted with two vectors, so one einsum over ($i, j, k$, segment) gives the field of the whole coil at every grid point. The loop radius swells and shrinks while the field in the $x$–$z$ plane and on the axis follow.

**Vector form**

$$\mathbf B(\mathbf x) = \frac{\mu_0 I}{4\pi}\oint\frac{d\mathbf l\times\mathbf r}{\lVert\mathbf r\rVert^3},\qquad \mathbf r = \mathbf x - \mathbf x_{\text{wire}}$$

**Index (tensor) form**

$$B_{gi} = \frac{\mu_0 I}{4\pi}\,\varepsilon_{ijk}\,dl_{sj}\,\frac{r_{gsk}}{\big(r_{gsl}\,r_{gsl}\big)^{3/2}},\qquad r_{gsk} = x_{gk} - x_{sk}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{gk}$ | `points` | (G, 3) | field point $g$ on the plane $y = 0$ |
| $x_{sk},\ dl_{sj}$ | `mid, dl` | (S, 3) | midpoint and direction vector of wire segment $s$ |
| $r_{gsk}$ | `points[:, None, :] - mid[None, :, :]` | (G, S, 3) | from every segment to every field point (broadcasting) |
| $\varepsilon_{ijk}$ | `levi_civita(dev)` | (3, 3, 3) | Levi-Civita symbol: the cross product as a tensor |
| $B_{gi}$ | `einsum('ijk,sj,gsk,gs->gi', eps, dl, r, inv_r3)` | (G, 3) | repeated $j, k$ ⇒ cross product; repeated $s$ ⇒ sum over the wire |
| $B_z(0, 0, z)$ | `loop_axis_field(z, R, z_loops, currents)` | (A,) | exact $\mu_0 I R^2/(2(R^2 + z^2)^{3/2})$ per loop, to check the sum |

SI units ($\mu_0 = 4\pi\times 10^{-7}$ T·m/A), field shown in μT. The loops lie in horizontal planes, so the $x$–$z$ plane cuts each one at two points (⊙ current out of the page, ⊗ into it). Helmholtz coils (spacing $R$, equal currents) make an almost uniform field between them; anti-Helmholtz coils (opposite currents) give a field that vanishes at the centre and grows linearly — the trap used for cold atoms. Grid points next to the wire use a small softening length so the field stays finite.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/em/biot_savart/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    eps = tensors.levi_civita(dev)  # (3, 3, 3)
    radii = tensors.radii(params["radius"], 72)

    result = solve.solve(params["layout"], radii, params["current"], params["n_segments"], params["n_grid"], eps, dev)

    error = (result["B_axis"][..., 2] - result["B_axis_exact"]).abs().max() / result["B_axis_exact"].abs().max()
    print(f"{params['n_segments']} segments per loop, {len(result['plane'])} field points per frame")
    print(f"centre field at R = {radii[0]:.2f} m: {result['B_axis_exact'][0].abs().max() * 1e6:.2f} μT")
    print(f"largest axis error, segments vs exact formula: {error:.2e} (relative)")
    return plot.animate(result, params["layout"], out_dir / "biot_savart.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/em/biot_savart/equation.py
r"""Physics → tensor translation.

The Biot–Savart law gives the magnetic field of a steady current I flowing along a wire,

    \mathbf B(\mathbf x) = \frac{\mu_0 I}{4\pi} \oint \frac{d\mathbf l \times \mathbf r}{\lVert \mathbf r \rVert^3},
    \qquad \mathbf r = \mathbf x - \mathbf x_{\text{wire}}

Cut the wire into S short straight segments s with midpoint x_{si} and direction vector dl_{si}.
For field points g the integral becomes a sum over s, and the cross product a Levi-Civita contraction:

    r_{gsk} = x_{gk} - x_{sk}
    B_{gi}  = \frac{\mu_0 I}{4\pi} \, \varepsilon_{ijk} \, dl_{sj} \, r_{gsk} \, / \, |r_{gs}|^3

j, k and s are repeated, so all three are summed — one einsum('ijk,sj,gsk,gs->gi') does the
cross product and adds up every segment of the wire.

Check: on the axis of a circular loop of radius R the field points along the axis with
B_z = μ0 I R² / (2 (R² + z²)^{3/2}).
"""

import math

import torch

MU0 = 4e-7 * math.pi  # T·m/A


def biot_savart(points, mid, dl, eps, current: float, softening: float = 0.0) -> torch.Tensor:
    r"""B_{gi} = (μ0 I/4π) ε_{ijk} dl_{sj} r_{gsk} / |r_{gs}|³.

    points: (G, 3), mid, dl: (S, 3), eps: (3, 3, 3) → (G, 3)
    softening (m) keeps the field finite for grid points that land right on the wire.
    """
    r = points[:, None, :] - mid[None, :, :]  # (G, S, 3) from each segment to each point
    r2 = torch.einsum("gsk,gsk->gs", r, r) + softening**2  # (G, S)   |r|²
    inv_r3 = r2**-1.5  # (G, S)
    return MU0 * current / (4 * math.pi) * torch.einsum("ijk,sj,gsk,gs->gi", eps, dl, r, inv_r3)


def loop_axis_field(z: torch.Tensor, R: float, z_loops: list[float], currents: list[float]) -> torch.Tensor:
    """Analytic B_z on the common axis of coaxial loops of radius R: Σ_l μ0 I_l R² / (2 (R² + (z − z_l)²)^{3/2})."""
    total = torch.zeros_like(z)
    for z_l, I_l in zip(z_loops, currents):
        total = total + MU0 * I_l * R**2 / (2 * (R**2 + (z - z_l) ** 2) ** 1.5)
    return total


def wire_field(r: torch.Tensor, current: float) -> torch.Tensor:
    """Analytic |B| = μ0 I / (2π r) at distance r from an infinitely long straight wire."""
    return MU0 * current / (2 * math.pi * r)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/em/biot_savart/tensors.py
"""Build the concrete tensors: Levi-Civita symbol, wire segments and field points in the x–z plane."""

import itertools
import math

import torch

EVEN = [(0, 1, 2), (1, 2, 0), (2, 0, 1)]  # cyclic permutations of (x, y, z)
EXTENT = 1.5  # the plot shows −EXTENT ≤ x, z ≤ EXTENT (metres)

# coaxial loops for each layout as (height in units of R, current direction)
LAYOUTS = {
    "loop": [(0.0, +1)],
    "helmholtz": [(-0.5, +1), (0.5, +1)],  # spacing R: the most uniform field between the coils
    "anti_helmholtz": [(-0.5, +1), (0.5, -1)],  # opposite currents: B = 0 in the middle
}


def levi_civita(device: torch.device) -> torch.Tensor:
    """ε_ijk of shape (3, 3, 3): +1 for even permutations, −1 for odd ones, 0 when an index repeats."""
    eps = torch.zeros(3, 3, 3, device=device)
    for i, j, k in itertools.permutations(range(3)):
        eps[i, j, k] = 1.0 if (i, j, k) in EVEN else -1.0
    return eps


def circle(R: float, z0: float, n: int, direction: int, device) -> tuple[torch.Tensor, torch.Tensor]:
    """A loop of radius R in the plane z = z0 cut into n segments: (midpoints (n, 3), dl (n, 3)).

    direction = +1 runs anticlockwise seen from +z (right-hand rule: B points up the axis).
    """
    phi = torch.linspace(0.0, 2 * math.pi, n + 1, device=device)  # n + 1 corners, the last equals the first
    corners = torch.stack([R * torch.cos(phi), R * torch.sin(phi), torch.full_like(phi, z0)], dim=-1)
    if direction < 0:
        corners = corners.flip(0)
    mid = 0.5 * (corners[1:] + corners[:-1])  # (n, 3)
    dl = corners[1:] - corners[:-1]  # (n, 3) segment vectors along the current
    return mid, dl


def coils(layout: str, R: float, n: int, device) -> tuple[torch.Tensor, torch.Tensor]:
    """All loops of a layout joined into one list of segments: (midpoints (S, 3), dl (S, 3))."""
    pieces = [circle(R, height * R, n, sign, device) for height, sign in LAYOUTS[layout]]
    return torch.cat([m for m, _ in pieces]), torch.cat([d for _, d in pieces])


def straight_wire(length: float, n: int, device) -> tuple[torch.Tensor, torch.Tensor]:
    """A straight wire along z from −length/2 to +length/2 in n segments: (midpoints (n, 3), dl (n, 3))."""
    z = torch.linspace(-length / 2, length / 2, n + 1, device=device)
    corners = torch.stack([torch.zeros_like(z), torch.zeros_like(z), z], dim=-1)
    return 0.5 * (corners[1:] + corners[:-1]), corners[1:] - corners[:-1]


def plane_xz(n: int, device) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (coordinates (N,), field points (N·N, 3)) on the plane y = 0; rows = z, columns = x."""
    xs = torch.linspace(-EXTENT, EXTENT, n, device=device)
    Z, X = torch.meshgrid(xs, xs, indexing="ij")
    points = torch.stack([X.reshape(-1), torch.zeros(n * n, device=device), Z.reshape(-1)], dim=-1)
    return xs, points


def z_axis(n: int, device) -> torch.Tensor:
    """n field points on the z axis, shape (n, 3)."""
    z = torch.linspace(-EXTENT, EXTENT, n, device=device)
    return torch.stack([torch.zeros_like(z), torch.zeros_like(z), z], dim=-1)


def radii(R_mean: float, n_steps: int) -> list[float]:
    """The loop radius at each animation step: it swings between 0.5 R_mean and 1.5 R_mean."""
    return [R_mean * (1 + 0.5 * math.sin(2 * math.pi * k / n_steps)) for k in range(n_steps)]


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/em/biot_savart/solve.py
"""For every loop radius: B on the x–z plane and on the axis, plus the analytic axis field for comparison."""

import torch

from . import tensors
from .equation import biot_savart, loop_axis_field


def solve(layout: str, radii: list[float], current: float, n_seg: int, n_grid: int, eps, device) -> dict:
    """Return a dict: B_plane (T, G, 3), B_axis (T, A, 3), B_axis_exact (T, A), plus the points used.

    Each radius is one call to biot_savart: every grid point and every wire segment in one einsum.
    """
    xs, plane = tensors.plane_xz(n_grid, device)  # (G, 3)
    axis = tensors.z_axis(200, device)  # (A, 3)
    h = xs[1] - xs[0]

    B_plane, B_axis, B_exact = [], [], []
    for R in radii:
        mid, dl = tensors.coils(layout, R, n_seg, device)  # (S, 3), (S, 3)
        B_plane.append(biot_savart(plane, mid, dl, eps, current, softening=0.5 * h))
        B_axis.append(biot_savart(axis, mid, dl, eps, current))
        heights = [z * R for z, _ in tensors.LAYOUTS[layout]]
        currents = [sign * current for _, sign in tensors.LAYOUTS[layout]]
        B_exact.append(loop_axis_field(axis[:, 2], R, heights, currents))

    return {
        "xs": xs,
        "plane": plane,
        "axis": axis,
        "radii": torch.tensor(radii),
        "B_plane": torch.stack(B_plane),
        "B_axis": torch.stack(B_axis),
        "B_axis_exact": torch.stack(B_exact),
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/em/biot_savart/plot.py
"""Animate the field: direction of B in the x–z plane, and B_z on the axis against the exact formula."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import frame_indices, save_gif

from .tensors import EXTENT, LAYOUTS

TESLA_TO_MICRO = 1e6


def animate(result: dict, layout: str, path: Path) -> Path:
    p = result["plane"].cpu().numpy()  # (G, 3)
    B = result["B_plane"].cpu().numpy()  # (T, G, 3)
    strength = np.linalg.norm(B, axis=-1)  # (T, G)
    Bx, Bz = B[..., 0] / strength, B[..., 2] / strength  # unit arrows: only the direction is drawn
    z = result["axis"][:, 2].cpu().numpy()  # (A,)
    Bz_axis = result["B_axis"][..., 2].cpu().numpy() * TESLA_TO_MICRO  # (T, A)
    Bz_exact = result["B_axis_exact"].cpu().numpy() * TESLA_TO_MICRO  # (T, A)
    radii = result["radii"].numpy()  # (T,)
    log_B = np.log10(strength * TESLA_TO_MICRO)
    lo, hi = np.quantile(log_B, [0.02, 0.98])

    fig, (ax, ax_axis) = plt.subplots(1, 2, figsize=(7, 4))

    # left: arrows coloured by log |B|; where each loop pierces the plane: ⊙ current out of the page, ⊗ into it
    arrows = ax.quiver(
        p[:, 0], p[:, 2], Bx[0], Bz[0], log_B[0], cmap="viridis", clim=(lo, hi), pivot="mid", scale=32, width=0.006
    )
    crossings = []
    for height, sign in LAYOUTS[layout]:
        for side in (-1, +1):
            symbol = "⊗" if side * sign > 0 else "⊙"
            crossings.append((side, height, ax.text(0, 0, symbol, ha="center", va="center", fontsize=14)))
    ax.axvline(0, color="#9ca3af", lw=0.8, ls=":")
    ax.set_xlim(-EXTENT, EXTENT)
    ax.set_ylim(-EXTENT, EXTENT)
    ax.set_aspect("equal")
    ax.grid(False)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("z (m)")
    title = ax.set_title("")

    # right: field along the dotted axis x = 0 — the summed segments against the textbook formula
    (exact_line,) = ax_axis.plot([], [], color="k", lw=1.5, label="exact")
    (numeric_line,) = ax_axis.plot([], [], "o", ms=3, color="C1", label="Σ over segments")
    ax_axis.set_xlim(-EXTENT, EXTENT)
    ax_axis.set_ylim(min(0, Bz_exact.min()) * 1.1, Bz_exact.max() * 1.1)
    ax_axis.set_xlabel("z on the axis (m)")
    ax_axis.set_ylabel("B_z (μT)")
    ax_axis.legend(loc="upper right", fontsize=8)
    ax_axis.set_title("field on the axis")

    every = 6  # draw every 6th numerical point so the exact curve stays visible

    def update(step: int):
        arrows.set_UVC(Bx[step], Bz[step], log_B[step])
        for side, height, label in crossings:
            label.set_position((side * radii[step], height * radii[step]))
        exact_line.set_data(z, Bz_exact[step])
        numeric_line.set_data(z[::every], Bz_axis[step, ::every])
        title.set_text(f"direction of B (colour: log₁₀|B|),  R = {radii[step]:.2f} m")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(radii), 72), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'layout': 'loop',
    'current': 10.0,
    'radius': 0.6,
    'n_segments': 60,
    'n_grid': 21
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.em.biot_savart")]:
    del sys.modules[name]
main = importlib.import_module("problems.em.biot_savart.main")
problem_dir = Path("problems/em/biot_savart")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
